# Does the V block split into two global halves?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import json

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display

pl.Config.set_tbl_rows(100)

def get_raw_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
v_cols = [c for c in df.columns if c.startswith("V")]


In [2]:
null_counts = [(c, df[c].null_count()) for c in v_cols]
df_nulls = pl.DataFrame(null_counts, schema=["column", "null_count"], orient="row")

family_sizes = (
    df_nulls.group_by("null_count")
    .agg(pl.len().alias("family_size"))
    .sort("family_size", descending=True)
)

fig = px.bar(family_sizes.to_pandas(), x="null_count", y="family_size", 
             title="Missingness Families: Size by Null Count",
             labels={"null_count": "Exact Null Count", "family_size": "Number of Columns in Family"},
             template="plotly_white")
fig.update_xaxes(type='category')
fig.show()

families = (
    df_nulls.group_by("null_count")
    .agg(pl.col("column").alias("cols"))
    .sort("null_count", descending=True)
    .to_dict(as_series=False)
)

table = (
    "| Null count | Family size | Columns |\n"
    "| --- | ---: | --- |\n"
    + "\n".join(
        f"| {null_count} | {len(cols)} | {', '.join(cols)} |"
        for null_count, cols in zip(families["null_count"], families["cols"])
    )
)
display(Markdown(table))

missingness_families = (
    df_nulls.group_by("null_count")
    .agg(pl.col("column").alias("cols"))
    .to_dict(as_series=False)
)
missingness_map = {c: cols for cols in missingness_families['cols'] for c in cols}


| Null count | Family size | Columns |
| --- | ---: | --- |
| 508595 | 18 | V138, V139, V140, V141, V142, V146, V147, V148, V149, V153, V154, V155, V156, V157, V158, V161, V162, V163 |
| 508589 | 11 | V143, V144, V145, V150, V151, V152, V159, V160, V164, V165, V166 |
| 508189 | 18 | V322, V323, V324, V325, V326, V327, V328, V329, V330, V331, V332, V333, V334, V335, V336, V337, V338, V339 |
| 460110 | 46 | V217, V218, V219, V223, V224, V225, V226, V228, V229, V230, V231, V232, V233, V235, V236, V237, V240, V241, V242, V243, V244, V246, V247, V248, V249, V252, V253, V254, V257, V258, V260, V261, V262, V263, V264, V265, V266, V267, V268, V269, V273, V274, V275, V276, V277, V278 |
| 450909 | 31 | V167, V168, V172, V173, V176, V177, V178, V179, V181, V182, V183, V186, V187, V190, V191, V192, V193, V196, V199, V202, V203, V204, V205, V206, V207, V211, V212, V213, V214, V215, V216 |
| 450721 | 19 | V169, V170, V171, V174, V175, V180, V184, V185, V188, V189, V194, V195, V197, V198, V200, V201, V208, V209, V210 |
| 449124 | 16 | V220, V221, V222, V227, V234, V238, V239, V245, V250, V251, V255, V256, V259, V270, V271, V272 |
| 279287 | 11 | V1, V2, V3, V4, V5, V6, V7, V8, V9, V10, V11 |
| 168969 | 18 | V35, V36, V37, V38, V39, V40, V41, V42, V43, V44, V45, V46, V47, V48, V49, V50, V51, V52 |
| 89164 | 20 | V75, V76, V77, V78, V79, V80, V81, V82, V83, V84, V85, V86, V87, V88, V89, V90, V91, V92, V93, V94 |
| 77096 | 22 | V53, V54, V55, V56, V57, V58, V59, V60, V61, V62, V63, V64, V65, V66, V67, V68, V69, V70, V71, V72, V73, V74 |
| 76073 | 23 | V12, V13, V14, V15, V16, V17, V18, V19, V20, V21, V22, V23, V24, V25, V26, V27, V28, V29, V30, V31, V32, V33, V34 |
| 1269 | 11 | V281, V282, V283, V288, V289, V296, V300, V301, V313, V314, V315 |
| 314 | 43 | V95, V96, V97, V98, V99, V100, V101, V102, V103, V104, V105, V106, V107, V108, V109, V110, V111, V112, V113, V114, V115, V116, V117, V118, V119, V120, V121, V122, V123, V124, V125, V126, V127, V128, V129, V130, V131, V132, V133, V134, V135, V136, V137 |
| 12 | 32 | V279, V280, V284, V285, V286, V287, V290, V291, V292, V293, V294, V295, V297, V298, V299, V302, V303, V304, V305, V306, V307, V308, V309, V310, V311, V312, V316, V317, V318, V319, V320, V321 |

In [3]:
def get_references_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "references").exists():
            return current / "references"
        current = current.parent
    return Path("../../references")

references_dir = get_references_dir()
with open(references_dir / "column-groups-v.json", "r") as f:
    col_groups_json = json.load(f)

subgroups = []
assigned_cols = set()
for block in col_groups_json['blocks']:
    for group in block['groups']:
        subgroups.append(group)
        assigned_cols.update(group)

unassigned = set(v_cols) - assigned_cols
for u in unassigned:
    subgroups.append([u])

table = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Total columns | {len(v_cols)} |\n"
    f"| Assigned in JSON | {len(assigned_cols)} |\n"
    f"| Unassigned | {len(unassigned)} |\n"
)
display(Markdown(table))


| Metric | Value |
| --- | ---: |
| Total columns | 339 |
| Assigned in JSON | 338 |
| Unassigned | 1 |


In [4]:
representatives = []
for group in subgroups:
    if len(group) == 1:
        representatives.append(group[0])
    else:
        uniques = [(c, df[c].n_unique()) for c in group]
        best_col = max(uniques, key=lambda x: x[1])[0]
        representatives.append(best_col)

total_cols = len(v_cols)
kept_cols = len(representatives)
dropped_cols = total_cols - kept_cols
reduction_pct = dropped_cols / total_cols * 100

fig = go.Figure(data=[go.Pie(labels=['Representatives Kept', 'Dropped'], 
                             values=[kept_cols, dropped_cols], hole=.4)])
fig.update_layout(title_text="Dimensionality Reduction via Selection", template="plotly_white")
fig.show()

table = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Columns in | {total_cols} |\n"
    f"| Representatives kept | **{kept_cols}** |\n"
    f"| Dropped | {dropped_cols} |\n"
    f"| Reduction | **{reduction_pct:.0f}%** |\n"
)
display(Markdown(table))


| Metric | Value |
| --- | ---: |
| Columns in | 339 |
| Representatives kept | **129** |
| Dropped | 210 |
| Reduction | **62%** |


In [5]:
import warnings

warnings.filterwarnings('ignore')

df_pd = df[v_cols].sample(n=120000, seed=42).to_pandas()
corr_matrix = df_pd.corr(method='pearson').abs()

results = []
measurable = 0
holds = 0
does_not_hold = 0

for group in subgroups:
    if len(group) <= 1:
        continue
        
    measurable += 1
    
    internal_corrs = []
    for i in range(len(group)):
        for j in range(i+1, len(group)):
            internal_corrs.append(corr_matrix.loc[group[i], group[j]])
    min_inside = min(internal_corrs)
    
    family_cols = set(missingness_map[group[0]])
    external_cols = list(family_cols - set(group))
    
    if not external_cols:
        holds += 1
        continue
        
    external_corrs = []
    for c_in in group:
        for c_out in external_cols:
            external_corrs.append(corr_matrix.loc[c_in, c_out])
    max_outside = max(external_corrs)
    
    if min_inside >= max_outside:
        holds += 1
    else:
        does_not_hold += 1
        results.append({
            "Group": "+".join(group) if len(group) <= 4 else f"{group[0]}+{group[1]}+{group[2]}…+{group[-1]}",
            "Size": len(group),
            "min inside": min_inside,
            "max outside": max_outside
        })

share = holds / measurable * 100 if measurable > 0 else 0

fig = px.bar(x=["Holds", "Does Not Hold"], y=[holds, does_not_hold], 
             title="Partition Validation Results",
             labels={'x': 'Status', 'y': 'Number of Groups'}, 
             color=["Holds", "Does Not Hold"], 
             color_discrete_map={"Holds": "green", "Does Not Hold": "red"},
             template="plotly_white")
fig.show()

summary = (
    f"| Metric | Value |\n"
    f"| --- | ---: |\n"
    f"| Measurable Groups | {measurable} |\n"
    f"| Hold | **{holds}** |\n"
    f"| Do not hold | **{does_not_hold}** |\n"
    f"| Share holding | **{share:.1f}%** |\n"
)
display(Markdown(summary))


| Metric | Value |
| --- | ---: |
| Measurable Groups | 93 |
| Hold | **63** |
| Do not hold | **30** |
| Share holding | **67.7%** |


In [6]:
df_offenders = pl.DataFrame(results).sort(pl.col("min inside") - pl.col("max outside"))
display(Markdown(df_offenders.head(5).to_pandas().to_markdown(index=False)))


| Group                |   Size |   min inside |   max outside |
|:---------------------|-------:|-------------:|--------------:|
| V334+V335+V336       |      3 |     0.439038 |      0.864207 |
| V205+V206            |      2 |     0.521578 |      0.882627 |
| V108+V109+V110+V114  |      4 |     0.443011 |      0.795885 |
| V186+V187+V190…+V199 |      8 |     0.570238 |      0.919474 |
| V263+V265+V264       |      3 |     0.627754 |      0.965687 |

### Global Block Split

Evaluating the observation that `V1-V100` and `V101-V339` form distinct macro-blocks with weak cross-correlation.

In [7]:
v1_v100 = [f"V{i}" for i in range(1, 101) if f"V{i}" in v_cols]
v101_v339 = [f"V{i}" for i in range(101, 340) if f"V{i}" in v_cols]

cross_corrs = corr_matrix.loc[v1_v100, v101_v339].values.flatten()
cross_corrs = cross_corrs[~np.isnan(cross_corrs)]

within_1_100 = corr_matrix.loc[v1_v100, v1_v100].values
np.fill_diagonal(within_1_100, np.nan)
within_1_100 = within_1_100[~np.isnan(within_1_100)]

within_101_339 = corr_matrix.loc[v101_v339, v101_v339].values
np.fill_diagonal(within_101_339, np.nan)
within_101_339 = within_101_339[~np.isnan(within_101_339)]

def stats(arr):
    if len(arr) == 0: return 0, 0
    return np.median(arr), np.mean(arr > 0.5) * 100

cross_med, cross_share = stats(cross_corrs)
w1_med, w1_share = stats(within_1_100)
w2_med, w2_share = stats(within_101_339)

summary2 = (
    f"| Scope | Median \\|r\\| | Share above 0.5 |\n"
    f"| --- | ---: | ---: |\n"
    f"| `V1–V100` against `V101–V339` | **{cross_med:.3f}** | **{cross_share:.1f}%** |\n"
    f"| within `V1–V100` | {w1_med:.3f} | {w1_share:.1f}% |\n"
    f"| within `V101–V339` | {w2_med:.3f} | {w2_share:.1f}% |\n"
)
display(Markdown(summary2))


| Scope | Median \|r\| | Share above 0.5 |
| --- | ---: | ---: |
| `V1–V100` against `V101–V339` | **0.033** | **2.9%** |
| within `V1–V100` | 0.097 | 16.5% |
| within `V101–V339` | 0.063 | 15.6% |
